# Classification de déchets avec PySpark MLlib

Ce notebook montre comment entraîner un modèle de Machine Learning pour identifier le type de déchet à partir de vos images préalablement traitées par `data_tansformation.py`.

In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, array_position, array_max
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, DoubleType
from datetime import datetime

spark = SparkSession.builder \
    .appName("GarbageClassificationML") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark



I0000 00:00:1774451414.865488  112960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774451414.873215  112960 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774451418.043834  112960 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774451427.331189  112960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

### Chargement et formatage des données
Keras nécessite que les features soient sous forme de `Veteurs`. Nos pixels sont stockés au format texte dans un .parquet

In [ ]:
def load_and_prepare_data(path, split):
    df = spark.read.parquet(path)
    label_col = f"y_{split}"
    feature_col = f"x_{split}"

    df = df.withColumn(
        "label",
        (array_position(col(label_col), array_max(col(label_col))) - 1)
    )

    return df.select("label", col(feature_col).alias("features")).dropna()

# Test
train_data = load_and_prepare_data("../data/train_data.parquet", "train")
test_data = load_and_prepare_data("../data/test_data.parquet", "test")
train_data.show(5)
test_data.show(5)


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/mathis/Desktop/Spark/Garbage-Classification/data/train_data.parquet.

In [ ]:
CONFIG = {
    "img_size": 64,
    "batch_size": 64,
    "epochs": 60,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
    "seed": 42
}

def data_augmentation():
    data_aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomZoom(0.05)
    ], name="data_augmentation")
    return data_aug

def create_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        data_augmentation(),
        tf.keras.layers.Rescaling(1.0 / 255.0),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ], name="CNN")
    
    return compile_model(model)


def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def spark_to_numpy(df):
    pdf = df.toPandas()
    y = pdf["label"].to_numpy(dtype=np.int32)
    X = np.stack([np.array(img, dtype=np.float32) for img in pdf["features"]])
    return X[..., np.newaxis], y

def save_model(model, name):
    filename = f"{name}.keras"
    save_path = os.path.join("..", "models", filename)
    model.save(save_path)
    print(f"Modèle sauvegardé : {save_path}")


def main():
    tf.keras.utils.set_random_seed(CONFIG["seed"])

    train_prepared = train_data.cache()

    train_split_df, val_df = train_prepared.randomSplit([0.9, 0.1], seed=CONFIG["seed"])

    X_train, y_train = spark_to_numpy(train_split_df)
    X_val, y_val = spark_to_numpy(val_df)

    input_shape = X_train.shape[1:]
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")
    print(f"Distribution val labels: {np.bincount(y_val)}")

    num_classes = int(max(y_train.max(), y_val.max()) + 1)

    class_counts = np.bincount(y_train, minlength=num_classes)
    class_weight = {
        i: float(len(y_train) / (num_classes * count))
        for i, count in enumerate(class_counts) if count > 0
    }

    model = create_cnn(input_shape, num_classes)

    model.summary()

    #Entraînement
    log_name = f"{model.name}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    current_log_path = os.path.join("..", "models", "logs", log_name)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True, mode="max"
            ),
            tf.keras.callbacks.TensorBoard(log_dir=current_log_path)
        ],
        verbose=2
    )
    save_model(model, f"final_{model.name}")
    spark.catalog.clearCache()


main()


Train: (6602, 64, 64, 1), Val: (722, 64, 64, 1)
Distribution val labels: [157 101 192  83 112  77]


E0000 00:00:1774362670.782151    7500 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,126,502 (8.11 MB)

 Trainable params: 2,126,310 (8.11 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/60
104/104 - 22s - 215ms/step - accuracy: 0.2569 - loss: 1.9405 - val_accuracy: 0.1150 - val_loss: 5.7567
Epoch 2/60
104/104 - 18s - 176ms/step - accuracy: 0.2775 - loss: 1.7117 - val_accuracy: 0.1150 - val_loss: 9.7298
Epoch 3/60
104/104 - 19s - 179ms/step - accuracy: 0.3031 - loss: 1.6713 - val_accuracy: 0.1150 - val_loss: 10.3741
Epoch 4/60
104/104 - 20s - 193ms/step - accuracy: 0.3223 - loss: 1.6551 - val_accuracy: 0.1537 - val_loss: 7.1244
Epoch 5/60
104/104 - 23s - 217ms/step - accuracy: 0.3291 - loss: 1.6305 - val_accuracy: 0.2119 - val_loss: 3.4577
Epoch 6/60
104/104 - 34s - 332ms/step - accuracy: 0.3272 - loss: 1.6208 - val_accuracy: 0.3006 - val_loss: 2.2090
Epoch 7/60
104/104 - 30s - 284ms/step - accuracy: 0.3354 - loss: 1.6134 - val_accuracy: 0.3449 - val_loss: 1.7649
Epoch 8/60
104/104 - 25s - 244ms/step - accuracy: 0.3540 - loss: 1.5900 - val_accuracy: 0.3587 - val_loss: 1.9087
Epoch 9/60
104/104 - 25s - 243ms/step - accuracy: 0.3617 - loss: 1.5675 - val_accuracy: